In [1]:
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, classification_report


df = pd.read_csv("test1.csv", encoding="utf-8-sig")
X = df[["dayoff", "nextdayoff", "jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec", "temperature", "rain","humidity"]]

#========================================
#이용객수 구간 범위 지정
#0-700명: 여유
#700~3000명 : 혼잡
#3000 이상 : 매우 혼잡
bins = [0,700, 3000, np.inf]
labels = [0, 1, 2]

y = pd.cut(
    df["visitors"],
    bins=bins,
    labels=labels,
    include_lowest=True
)
y = y.astype(int)

#========================================
# 요일별 순서로 되어있기 때문에 랜덤으로 분리
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]


#========================================
# 캣부스트
models = {
   "CatBoost": CatBoostClassifier(
        iterations=1164,
        depth=8,
        learning_rate=0.04585878172511418,
        l2_leaf_reg=8.928840273173956,
        bagging_temperature=0.9065380504018735,
        random_strength=0.051733258645387664,
        border_count=49,
        random_state=42,
        verbose = 0
       )
}

#========================================
#모델 학습 후 평가
results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred = np.asarray(y_pred).reshape(-1)

    f1 = f1_score(y_test, y_pred, average="weighted")       
    results.append({"Model": name, "f1": f1})
    predictions[name] = y_pred
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=[
                "0~700명",
                "700~3000명",
                "3001명 이상"
            ]
        )
    )

results_df = (
    pd.DataFrame(results)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

pd.options.display.float_format = "{:.2f}".format
print(results_df)

#model.save_model("Oworld_catboost_test1.cbm")

              precision    recall  f1-score   support

      0~700명       0.80      0.82      0.81       102
   700~3000명       0.70      0.74      0.72        85
    3001명 이상       0.94      0.81      0.87        57

    accuracy                           0.79       244
   macro avg       0.81      0.79      0.80       244
weighted avg       0.80      0.79      0.79       244

      Model   f1
0  CatBoost 0.79
